In [ ]:
shared_root <- normalizePath(Sys.getenv("REGA_SHARED_ROOT", unset = normalizePath("..")))
library(tidyverse)

In [ ]:
# ============================================================
# 1. Paths
# ============================================================

# Cross-cell-type / region-wise shared H matrices
H_ROOT <- file.path(shared_root, "part3_process_REGA_output/09extracted_matrices_region_shareH")

# Region-consistent metadata used by the current shared-model fit.
# Unchanged regions are linked to their existing shared-model inputs.
METADATA_ROOT <- file.path(shared_root, "part1_prepare_REGA/01region_consistent_metadata")

# Output directory
# OUT_DIR <- file.path(
#   H_ROOT,
#   "selected_cross_celltype_module_activity_boxplots"
# )
OUT_DIR <- file.path(shared_root, "part3_process_REGA_output/figs/module_boxplots")

dir.create(
  OUT_DIR,
  recursive = TRUE,
  showWarnings = FALSE
)

In [ ]:
# ============================================================
# 2. Selected cross-cell-type modules
# ============================================================

SELECTED_MODULES <- readLines(file.path(shared_root, "part2_run_REGA/08pathology_tests/selected_module.txt"))

SELECTED_MODULES


# ============================================================
# 3. Pathology colors
# ============================================================

pathology_colors <- c(
  "nonAD"   = "#66AABB",
  "earlyAD" = "#EEAA55",
  "lateAD"  = "#CC6677"
)


In [ ]:
# ============================================================
# 4. Read region-level metadata
#
# For example, for AG:
#   Ast_AG_region_consistent_metadata.tsv
#   Exc_AG_region_consistent_metadata.tsv
#   Inh_AG_region_consistent_metadata.tsv
#   Mic_Immune_AG_region_consistent_metadata.tsv
#   Oli_AG_region_consistent_metadata.tsv
#   OPC_AG_region_consistent_metadata.tsv
#
# are combined to get the union of individuals.
#
# Vasc is excluded because the H matrices were generated
# using region_shareH.
# ============================================================

read_region_metadata <- function(region) {

  metadata_files <- list.files(
    METADATA_ROOT,
    pattern = paste0("_", region, "_region_consistent_metadata\\.tsv$"),
    full.names = TRUE
  )

  # Exclude Vasc metadata
  metadata_files <- metadata_files[
    !grepl(
      paste0("(^|/)", "Vasc_", region, "_region_consistent_metadata\\.tsv$"),
      metadata_files
    )
  ]

  if (length(metadata_files) == 0) {
    stop(
      "No non-Vasc metadata files found for region: ",
      region
    )
  }

  message(
    "\nRegion ", region,
    ": using ", length(metadata_files),
    " metadata files"
  )

  message(
    paste0("  ", basename(metadata_files), collapse = "\n")
  )

  metadata <- map_dfr(
    metadata_files,
    function(f) {

      md <- read.delim(
        f,
        header = TRUE,
        sep = "\t",
        check.names = FALSE,
        stringsAsFactors = FALSE
      )

      required_cols <- c(
        "ROSMAP_IndividualID",
        "Pathology"
      )

      missing_cols <- setdiff(
        required_cols,
        colnames(md)
      )

      if (length(missing_cols) > 0) {
        stop(
          "Missing columns in ",
          f,
          ": ",
          paste(missing_cols, collapse = ", ")
        )
      }

      md %>%
        select(
          ROSMAP_IndividualID,
          Pathology
        )
    }
  ) %>%
    filter(
      !is.na(ROSMAP_IndividualID),
      !is.na(Pathology)
    ) %>%
    distinct()


  # ----------------------------------------------------------
  # Check whether the same individual has conflicting
  # pathology labels across cell types
  # ----------------------------------------------------------

  pathology_conflict <- metadata %>%
    group_by(ROSMAP_IndividualID) %>%
    summarise(
      n_pathology = n_distinct(Pathology),
      pathology_labels = paste(
        sort(unique(Pathology)),
        collapse = ","
      ),
      .groups = "drop"
    ) %>%
    filter(n_pathology > 1)


  if (nrow(pathology_conflict) > 0) {

    print(pathology_conflict)

    stop(
      "Conflicting Pathology labels detected for region ",
      region
    )
  }


  # One row per individual
  metadata <- metadata %>%
    distinct(
      ROSMAP_IndividualID,
      .keep_all = TRUE
    ) %>%
    filter(
      Pathology %in%
        c("nonAD", "earlyAD", "lateAD")
    )


  message(
    "  Union individuals with valid pathology: ",
    nrow(metadata)
  )

  return(metadata)
}



In [ ]:
# ============================================================
# 5. Function to read one cross-cell-type module
# ============================================================

read_one_module <- function(selected_name) {

  # ----------------------------------------------------------
  # Example:
  #
  # selected_name = "AG_M13"
  #
  # region = "AG"
  # module = "M13"
  # ----------------------------------------------------------

  region <- str_extract(
    selected_name,
    "^[^_]+"
  )

  module <- str_extract(
    selected_name,
    "M[0-9]+$"
  )


  message(
    "\n========================================"
  )
  message(
    "Reading ", selected_name
  )
  message(
    "Region: ", region,
    " | Module: ", module
  )


  # ----------------------------------------------------------
  # H matrix
  #
  # Example:
  #
  # .../AG/AG_H_module_by_sample.tsv
  # ----------------------------------------------------------

  H_file <- file.path(
    H_ROOT,
    region,
    paste0(
      region,
      "_H_module_by_sample.tsv"
    )
  )


  if (!file.exists(H_file)) {
    stop(
      "H file not found:\n",
      H_file
    )
  }


  # ----------------------------------------------------------
  # Read H
  #
  # read_tsv safely preserves the first module column,
  # even though its header may be blank in the TSV.
  # ----------------------------------------------------------

  H <- suppressMessages(
    readr::read_tsv(
      H_file,
      show_col_types = FALSE
    )
  )


  # First column = M1, M2, M3, ...
  colnames(H)[1] <- "Module"

  H$Module <- as.character(
    H$Module
  )


  if (!module %in% H$Module) {

    stop(
      module,
      " was not found in:\n",
      H_file
    )
  }


  # ----------------------------------------------------------
  # Extract one module and convert samples to rows
  # ----------------------------------------------------------

  H_long <- H %>%
    filter(
      Module == module
    ) %>%
    pivot_longer(
      cols = -Module,
      names_to = "ROSMAP_IndividualID",
      values_to = "module_activity"
    ) %>%
    mutate(
      module_activity =
        as.numeric(module_activity)
    )


  n_H <- nrow(H_long)

  message(
    "Individuals in H matrix: ",
    n_H
  )


  # ----------------------------------------------------------
  # Region metadata
  # ----------------------------------------------------------

  metadata <- read_region_metadata(
    region
  )


  # ----------------------------------------------------------
  # Match H individuals to pathology metadata
  # ----------------------------------------------------------

  result <- H_long %>%
    inner_join(
      metadata,
      by = "ROSMAP_IndividualID"
    ) %>%
    filter(
      Pathology %in%
        c(
          "nonAD",
          "earlyAD",
          "lateAD"
        )
    ) %>%
    mutate(
      selected_module = selected_name,
      region = region,
      module = module,

      Pathology = factor(
        Pathology,
        levels = c(
          "nonAD",
          "earlyAD",
          "lateAD"
        )
      )
    )


  message(
    "Individuals matched to pathology: ",
    nrow(result)
  )


  # ----------------------------------------------------------
  # Report unmatched H individuals
  # ----------------------------------------------------------

  unmatched <- H_long %>%
    anti_join(
      metadata,
      by = "ROSMAP_IndividualID"
    )


  if (nrow(unmatched) > 0) {

    stop(
      selected_name,
      ": ",
      nrow(unmatched),
      " H individuals were not found ",
      "in region metadata."
    )
  }


  # ----------------------------------------------------------
  # Show pathology sample counts
  # ----------------------------------------------------------

  print(
    result %>%
      count(Pathology)
  )


  return(result)
}


# ============================================================
# 6. Read all selected modules
# ============================================================

plot_data <- map_dfr(
  SELECTED_MODULES,
  read_one_module
)


if (nrow(plot_data) == 0) {
  stop(
    "No H values could be matched to metadata."
  )
}


# ============================================================
# 7. Make sure every requested module was found
# ============================================================

found_modules <- unique(
  plot_data$selected_module
)

missing_modules <- setdiff(
  SELECTED_MODULES,
  found_modules
)


if (length(missing_modules) > 0) {

  stop(
    "These selected modules were not successfully loaded: ",
    paste(
      missing_modules,
      collapse = ", "
    )
  )
}


# ============================================================
# 8. Keep desired x-axis order
# ============================================================

plot_data <- plot_data %>%
  mutate(
    selected_module = factor(
      selected_module,
      levels = SELECTED_MODULES
    )
  )


# ============================================================
# 9. Save plotting table
# ============================================================

write.table(
  plot_data,
  file.path(
    OUT_DIR,
    "cross_celltype_selected_module_activity_plot_data.tsv"
  ),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)


# ============================================================
# 10. Save sample-count table
# ============================================================

sample_count <- plot_data %>%
  count(
    selected_module,
    Pathology,
    name = "n_individuals"
  )


write.table(
  sample_count,
  file.path(
    OUT_DIR,
    "cross_celltype_selected_module_pathology_sample_counts.tsv"
  ),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

sample_count


# ============================================================
# 11. Calculate visible boxplot range
#
# Use whiskers rather than extreme hidden outliers.
# ============================================================

whisker_limits <- plot_data %>%
  group_by(
    selected_module,
    Pathology
  ) %>%
  summarise(

    whisker_low =
      boxplot.stats(
        module_activity
      )$stats[1],

    whisker_high =
      boxplot.stats(
        module_activity
      )$stats[5],

    .groups = "drop"
  )


y_min <- min(
  whisker_limits$whisker_low,
  na.rm = TRUE
)

y_max <- max(
  whisker_limits$whisker_high,
  na.rm = TRUE
)


y_padding <- (
  y_max - y_min
) * 0.06


if (
  !is.finite(y_padding) ||
  y_padding == 0
) {
  y_padding <- 0.1
}


# ============================================================
# 12. Plot
# ============================================================

p <- ggplot(
  plot_data,
  aes(
    x = selected_module,
    y = module_activity,
    fill = Pathology
  )
) +

  geom_boxplot(
    position =
      position_dodge(
        width = 0.80
      ),
    width = 0.68,
    outlier.shape = NA,
    linewidth = 0.55
  ) +

  scale_fill_manual(
    values = pathology_colors,
    drop = FALSE
  ) +

  scale_y_continuous(
    expand = expansion(
      mult = c(0, 0)
    )
  ) +

  coord_cartesian(
    ylim = c(
      y_min - y_padding,
      y_max + y_padding
    ),
    clip = "on"
  ) +

  labs(
    x = NULL,
    y = "Module activity",
    fill = "Pathology"
  ) +

  theme_classic(
    base_size = 18
  ) +

  theme(

    axis.text.x = element_text(
      angle = 90,
      hjust = 1,
      vjust = 0.5,
      color = "black",
      size = 15
    ),

    axis.text.y = element_text(
      color = "black",
      size = 16
    ),

    axis.title.y = element_text(
      face = "bold",
      size = 20
    ),

    legend.text = element_text(
      size = 16
    ),

    legend.title = element_text(
      face = "bold",
      size = 18
    ),

    legend.position = "top",

    plot.margin = margin(
      6,
      12,
      6,
      8
    )
  )


p


# ============================================================
# 13. Save plots
# ============================================================

plot_width <- max(
  8,
  length(SELECTED_MODULES) * 0.95
)


ggsave(
  file.path(
    OUT_DIR,
    "cross_celltype_selected_module_activity_boxplots.pdf"
  ),
  p,
  width = plot_width,
  height = 6.5,
  limitsize = FALSE
)


ggsave(
  file.path(
    OUT_DIR,
    "cross_celltype_selected_module_activity_boxplots.png"
  ),
  p,
  width = plot_width,
  height = 6.5,
  dpi = 300,
  limitsize = FALSE
)

In [ ]:
ggsave(
  file.path(
    OUT_DIR,
    "cross_celltype_selected_module_activity_boxplots4.pdf"
  ),
  p,
  width = plot_width,
  height = 4,
  limitsize = FALSE
)
